# Camada Bronze — Landing-Zone (CSV) → Bronze (Delta Lake)

Este notebook implementa a **segunda etapa** da Arquitetura Medalhão:

1. Lê os CSVs gerados pelo notebook `01` no bucket `landing-zone` do MinIO
2. Aplica **tipagem forte** (schema enforcement) em cada tabela
3. Adiciona **metadados de auditoria** (`_bronze_loaded_at`, `_bronze_source_file`)
4. Grava no formato **Delta Lake** no bucket `bronze` do MinIO
5. Valida a contagem de registros entre landing e bronze

**Pré-requisito**: notebooks `00` e `01` executados e containers no ar (`docker compose up -d`).

## 1. Configuração de ambiente e variáveis

In [10]:
import os
from pathlib import Path
from datetime import datetime

from dotenv import load_dotenv

# Carrega .env a partir da raiz do projeto (pai da pasta notebook/)
_root = Path.cwd()
if _root.name == "notebook":
    _root = _root.parent
load_dotenv(_root / ".env")

required_minio = ["MINIO_ROOT_USER", "MINIO_ROOT_PASSWORD", "MINIO_S3_ENDPOINT"]
missing = [k for k in required_minio if not os.getenv(k)]
if missing:
    raise RuntimeError(f"Defina no .env: {', '.join(missing)}")

MINIO_ENDPOINT = os.getenv("MINIO_S3_ENDPOINT")
MINIO_ACCESS_KEY = os.getenv("MINIO_ROOT_USER")
MINIO_SECRET_KEY = os.getenv("MINIO_ROOT_PASSWORD")

LANDING_BUCKET = "landing-zone"
BRONZE_BUCKET = "bronze"

print("MinIO endpoint:", MINIO_ENDPOINT)
print("Landing bucket:", LANDING_BUCKET)
print("Bronze bucket: ", BRONZE_BUCKET)

MinIO endpoint: http://127.0.0.1:9000
Landing bucket: landing-zone
Bronze bucket:  bronze


## 2. Inicialização do SparkSession com Delta Lake e S3A (MinIO)

In [11]:
from pyspark.sql import SparkSession

# Pacotes compatíveis com Spark 3.5.x (Hadoop 3.3.4 interno)
packages = (
    "io.delta:delta-spark_2.12:3.2.0,"
    "org.apache.hadoop:hadoop-aws:3.3.4,"
    "com.amazonaws:aws-java-sdk-bundle:1.12.262"
)

spark = (
    SparkSession.builder.appName("landing-to-bronze-delta")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
    .config("spark.jars.packages", packages)
    # MinIO via S3A
    .config("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider",
    )
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark inicializado:", spark.version)

Spark inicializado: 3.5.3


## 3. Definição dos schemas das tabelas do BibliotecaDb

Na camada Bronze aplicamos **tipagem forte** para garantir consistência desde o início do pipeline.
Cada tabela recebe seu `StructType` correspondente ao schema do SQL Server.

In [12]:
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    DateType,
    DecimalType,
    BooleanType,
)

# ── Schemas das 6 tabelas do BibliotecaDb ──────────────────────────────

schema_categoria = StructType([
    StructField("id_categoria", IntegerType(), False),
    StructField("nome", StringType(), False),
    StructField("descricao", StringType(), True),
])

schema_autor = StructType([
    StructField("id_autor", IntegerType(), False),
    StructField("nome", StringType(), False),
    StructField("nacionalidade", StringType(), True),
    StructField("data_nascimento", DateType(), True),
])

schema_livro = StructType([
    StructField("id_livro", IntegerType(), False),
    StructField("titulo", StringType(), False),
    StructField("isbn", StringType(), True),
    StructField("ano_publicacao", IntegerType(), True),
    StructField("id_categoria", IntegerType(), False),
    StructField("id_autor", IntegerType(), False),
])

schema_membro = StructType([
    StructField("id_membro", IntegerType(), False),
    StructField("nome", StringType(), False),
    StructField("email", StringType(), False),
    StructField("telefone", StringType(), True),
    StructField("data_cadastro", DateType(), False),
])

schema_emprestimo = StructType([
    StructField("id_emprestimo", IntegerType(), False),
    StructField("id_livro", IntegerType(), False),
    StructField("id_membro", IntegerType(), False),
    StructField("data_emprestimo", DateType(), False),
    StructField("data_devolucao_prevista", DateType(), False),
    StructField("data_devolucao_real", DateType(), True),
    StructField("status", StringType(), False),
])

schema_multa = StructType([
    StructField("id_multa", IntegerType(), False),
    StructField("id_emprestimo", IntegerType(), False),
    StructField("valor", DecimalType(10, 2), False),
    StructField("pago", BooleanType(), False),
    StructField("data_geracao", DateType(), False),
])

# Mapeamento: nome da pasta na landing-zone → schema tipado
TABELAS = {
    "dbo_Categoria":  schema_categoria,
    "dbo_Autor":      schema_autor,
    "dbo_Livro":      schema_livro,
    "dbo_Membro":     schema_membro,
    "dbo_Emprestimo": schema_emprestimo,
    "dbo_Multa":      schema_multa,
}

print(f"{len(TABELAS)} tabelas configuradas para ingestão na Bronze:")
for nome in TABELAS:
    print(f"  → {nome}")

6 tabelas configuradas para ingestão na Bronze:
  → dbo_Categoria
  → dbo_Autor
  → dbo_Livro
  → dbo_Membro
  → dbo_Emprestimo
  → dbo_Multa


## 4. Extração Landing → Bronze (CSV → Delta Lake)

Para cada tabela:
- Lê o CSV da `landing-zone` com o schema definido
- Adiciona colunas de auditoria (`_bronze_loaded_at`, `_bronze_source_file`)
- Grava em formato **Delta Lake** no bucket `bronze` com `mode("overwrite")`

In [13]:
from pyspark.sql import functions as F

load_ts = datetime.now().isoformat()

success = []
errors = []
total_rows = 0

for folder, schema in TABELAS.items():
    landing_path = f"s3a://{LANDING_BUCKET}/{folder}/"
    bronze_path = f"s3a://{BRONZE_BUCKET}/{folder}/"
    print(f"Processando {folder}")
    print(f"  Landing: {landing_path}")
    print(f"  Bronze:  {bronze_path}")

    try:
        # Lê CSV da landing-zone com schema tipado
        df = (
            spark.read
            .option("header", True)
            .option("encoding", "UTF-8")
            .schema(schema)
            .csv(landing_path)
        )

        # Adiciona metadados de auditoria (rastreabilidade)
        df_bronze = (
            df
            .withColumn("_bronze_loaded_at", F.lit(load_ts))
            .withColumn("_bronze_source_file", F.input_file_name())
        )

        n = df_bronze.count()
        total_rows += n

        # Grava em Delta Lake
        (
            df_bronze.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .save(bronze_path)
        )

        success.append({"tabela": folder, "linhas": n, "path": bronze_path})
        print(f"  ✔ {n} linhas gravadas em Delta")

    except Exception as exc:
        errors.append({"tabela": folder, "erro": str(exc)})
        print(f"  ✘ ERRO — {exc}")

print(f"\n{'='*50}")
print(f"Resumo da ingestão Landing → Bronze")
print(f"{'='*50}")
print(f"Sucesso:  {len(success)}")
print(f"Com erro: {len(errors)}")
print(f"Total de linhas ingeridas: {total_rows}")
if errors:
    print("\nFalhas:")
    for e in errors:
        print(f"  {e['tabela']}: {e['erro']}")

Processando dbo_Categoria
  Landing: s3a://landing-zone/dbo_Categoria/
  Bronze:  s3a://bronze/dbo_Categoria/


  ✔ 5 linhas gravadas em Delta
Processando dbo_Autor
  Landing: s3a://landing-zone/dbo_Autor/
  Bronze:  s3a://bronze/dbo_Autor/


  ✔ 10 linhas gravadas em Delta
Processando dbo_Livro
  Landing: s3a://landing-zone/dbo_Livro/
  Bronze:  s3a://bronze/dbo_Livro/
  ✔ 20 linhas gravadas em Delta
Processando dbo_Membro
  Landing: s3a://landing-zone/dbo_Membro/
  Bronze:  s3a://bronze/dbo_Membro/
  ✔ 15 linhas gravadas em Delta
Processando dbo_Emprestimo
  Landing: s3a://landing-zone/dbo_Emprestimo/
  Bronze:  s3a://bronze/dbo_Emprestimo/
  ✔ 30 linhas gravadas em Delta
Processando dbo_Multa
  Landing: s3a://landing-zone/dbo_Multa/
  Bronze:  s3a://bronze/dbo_Multa/
  ✔ 8 linhas gravadas em Delta

Resumo da ingestão Landing → Bronze
Sucesso:  6
Com erro: 0
Total de linhas ingeridas: 88


## 5. Validação — Leitura Delta e contagem

Relê cada tabela Delta do bucket `bronze` para validar que os dados foram gravados corretamente.

In [14]:
print("Validação: leitura das tabelas Delta no bucket bronze")
print(f"{'='*60}")
print(f"{'Tabela':<20} {'Landing (CSV)':<15} {'Bronze (Delta)':<15} {'Status'}")
print(f"{'-'*60}")

all_ok = True
for folder, schema in TABELAS.items():
    landing_path = f"s3a://{LANDING_BUCKET}/{folder}/"
    bronze_path = f"s3a://{BRONZE_BUCKET}/{folder}/"

    try:
        count_landing = spark.read.option("header", True).schema(schema).csv(landing_path).count()
        count_bronze = spark.read.format("delta").load(bronze_path).count()
        status = "✔ OK" if count_landing == count_bronze else "✘ DIVERGENTE"
        if count_landing != count_bronze:
            all_ok = False
        print(f"{folder:<20} {count_landing:<15} {count_bronze:<15} {status}")
    except Exception as exc:
        all_ok = False
        print(f"{folder:<20} {'?':<15} {'?':<15} ✘ ERRO: {exc}")

print(f"{'-'*60}")
if all_ok:
    print("\n✔ Todas as contagens conferem entre Landing e Bronze.")
else:
    print("\n✘ Há divergências — verifique os erros acima.")

Validação: leitura das tabelas Delta no bucket bronze
Tabela               Landing (CSV)   Bronze (Delta)  Status
------------------------------------------------------------
dbo_Categoria        5               5               ✔ OK
dbo_Autor            10              10              ✔ OK
dbo_Livro            20              20              ✔ OK
dbo_Membro           15              15              ✔ OK
dbo_Emprestimo       30              30              ✔ OK
dbo_Multa            8               8               ✔ OK
------------------------------------------------------------

✔ Todas as contagens conferem entre Landing e Bronze.


## 6. Amostra dos dados Bronze

Exibe as primeiras linhas de cada tabela Delta para verificação visual.

In [15]:
for folder in TABELAS:
    bronze_path = f"s3a://{BRONZE_BUCKET}/{folder}/"
    print(f"\n{'─'*60}")
    print(f"Tabela: {folder}")
    print(f"{'─'*60}")
    df = spark.read.format("delta").load(bronze_path)
    df.printSchema()
    df.show(5, truncate=False)


────────────────────────────────────────────────────────────
Tabela: dbo_Categoria
────────────────────────────────────────────────────────────
root
 |-- id_categoria: integer (nullable = true)
 |-- nome: string (nullable = true)
 |-- descricao: string (nullable = true)
 |-- _bronze_loaded_at: string (nullable = true)
 |-- _bronze_source_file: string (nullable = true)

+------------+-----------------+------------------------------------------------------------------------------------+--------------------------+-----------------------------------------------------------------------------------------+
|id_categoria|nome             |descricao                                                                           |_bronze_loaded_at         |_bronze_source_file                                                                      |
+------------+-----------------+------------------------------------------------------------------------------------+--------------------------+-------------

## 7. Histórico Delta (versionamento)

O Delta Lake mantém um log transacional de cada operação. Vamos verificar o histórico de uma tabela.

In [16]:
from delta.tables import DeltaTable

# Exemplo com a tabela Emprestimo (maior volume)
delta_path = f"s3a://{BRONZE_BUCKET}/dbo_Emprestimo/"
dt = DeltaTable.forPath(spark, delta_path)

print("Histórico de versões — dbo_Emprestimo (Bronze):")
dt.history().select("version", "timestamp", "operation", "operationMetrics").show(truncate=False)

Histórico de versões — dbo_Emprestimo (Bronze):
+-------+-------------------+---------+------------------------------------------------------------+
|version|timestamp          |operation|operationMetrics                                            |
+-------+-------------------+---------+------------------------------------------------------------+
|1      |2026-05-03 15:28:04|WRITE    |{numFiles -> 1, numOutputRows -> 30, numOutputBytes -> 4085}|
|0      |2026-05-03 15:24:11|WRITE    |{numFiles -> 1, numOutputRows -> 30, numOutputBytes -> 4085}|
+-------+-------------------+---------+------------------------------------------------------------+



## 8. Listagem de objetos no bucket bronze

In [17]:
def list_s3a_prefix(spark_session, uri: str, indent: str = "") -> None:
    """Lista caminhos sob um prefixo s3a usando a API Hadoop já configurada no Spark."""
    jvm = spark_session.sparkContext._jvm
    hadoop_conf = spark_session.sparkContext._jsc.hadoopConfiguration()
    path = jvm.org.apache.hadoop.fs.Path(uri)
    fs = path.getFileSystem(hadoop_conf)
    if not fs.exists(path):
        print(f"{indent}(prefixo inexistente ou vazio)")
        return
    statuses = fs.listStatus(path)
    for st in statuses:
        p = st.getPath().toString()
        if st.isDirectory():
            print(f"{indent}{p}/")
            list_s3a_prefix(spark_session, p + "/", indent + "  ")
        else:
            size_kb = st.getLen() / 1024
            print(f"{indent}{p}  ({size_kb:.1f} KB)")


print(f"Objetos em s3a://{BRONZE_BUCKET}/")
list_s3a_prefix(spark, f"s3a://{BRONZE_BUCKET}/")

Objetos em s3a://bronze/
s3a://bronze/dbo_Autor/
  s3a://bronze/dbo_Autor/part-00000-791661b8-b533-4489-9bf8-c6569e8305a0-c000.snappy.parquet  (2.9 KB)
  s3a://bronze/dbo_Autor/part-00000-e2a6db74-73ad-405f-a848-67c87cd5b011-c000.snappy.parquet  (2.9 KB)
  s3a://bronze/dbo_Autor/_delta_log/
    s3a://bronze/dbo_Autor/_delta_log/00000000000000000000.json  (1.9 KB)
    s3a://bronze/dbo_Autor/_delta_log/00000000000000000001.json  (1.4 KB)
    s3a://bronze/dbo_Autor/_delta_log/_commits/
s3a://bronze/dbo_Categoria/
  s3a://bronze/dbo_Categoria/part-00000-a2f8e1b9-33ff-4d39-9baf-22776a98db4f-c000.snappy.parquet  (3.0 KB)
  s3a://bronze/dbo_Categoria/part-00000-ac3b1fc8-9ea9-492b-89c3-4a120c17da82-c000.snappy.parquet  (3.0 KB)
  s3a://bronze/dbo_Categoria/_delta_log/
    s3a://bronze/dbo_Categoria/_delta_log/00000000000000000000.json  (1.8 KB)
    s3a://bronze/dbo_Categoria/_delta_log/00000000000000000001.json  (1.3 KB)
    s3a://bronze/dbo_Categoria/_delta_log/_commits/
s3a://bronze/dbo_Empr

In [18]:
spark.stop()
print("SparkSession encerrada.")

SparkSession encerrada.
